# Wstępna Eksploracyjna Analiza Danych ze Steam
Celem tej sekcji jest ustalenie wstępnych wymagań wobec zbioru danych przed jego pełnym pobraniem ze Steam API.

W projekcie celem jest stworzenie algorytmu wykrywającego sentyment oceny gry, uwzględniając sarkzam.

Wytrenowanie skutecznego modelu będzie wymagało przekroju różnych opinii, z różnych gatunków gier, o różnych wymogach technicznych. Powody opinii negatywnej czy sarkastycznej mogą wynikać zarówno z problemów z fabułą i mechaniką gry, ale również z braku optymalizacji gry dla słabszego sprzętu. Algorytm wytrenowany wyłącznie na recenzjach najpopularniejszych gier mogłoby skutkować z tendencyjnością algorytmu i wysoką liczebnością klasy False Negative.

Dodatkowo zdaniem autora najpopularniejsze gry mogą częściej mieć problemy z optymalizacją niż z samą rozgrywką, niż przeciętna gra. Gracze chętniej spróbują zagrać w ciekawą grę, która jest niezoptymalizowana na ich sprzęt, z nadzieją, że twórca gry po pewnym czasie wypuści aktualizację, która ustabilizuje ilość klatek na sekundę na ich sprzęcie. Natomiast, jeśli gra jest dobrze zoptymalizowana ale nudna gra nigdy nie trafi do listy najpopularniejszych gier.

Osobnym wyzwaniem jest ilość opinii w języku polskim. Sarkazm jest zjawiskiem bardzo specficznym dla każdego języka, dlatego do trenowania modelu autor wybierze wyłącznie opinie w języku polskim. Tym sposobem autor będzie mógł na koniec podejrzeć wyniki algorytmu i samodzielnie ocenić jego poprawne funkcjonowanie. Ograniczenie językowe oznacza, że skupienie się wyłącznie na mało popularnych grach może skutkować trudnością z uzbieraniem odpowiedniej ilości opinii i znacznie większą ilością zapytań do Steam API.

Zważając na powyższe ograniczenia autor zamierza najpierw podzielić dużą populację najczęściej granych gier podzielić na grupy według ich kategorii oceny Steam, a następnie z każdej grupy losowo wybrać podobną liczbę gier, aby cały zbiór recenzji posiadał około **??** tysięcy opinii.

## Przegląd najpopularniejszych gier na platformie Steam

In [33]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import re

In [34]:
url = "https://store.steampowered.com/search/?supportedlang=polish%2Cenglish&category1=998&ndl=1"
    
# Udajemy prawdziwą przeglądarkę (Header), żeby Steam nas nie zablokował
headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36'
}

# 4. Wywołujemy zapytanie przekazując ZARÓWNO nagłówki, jak i ciasteczka
response = requests.get(url, headers=headers)

# Jeśli strona nie zwraca JSON-a, bierzemy surowy tekst HTML
html_content = response.text

print(html_content)

<!DOCTYPE html>
<html class=" responsive DesktopUI" lang="en"  >
<head>
	<meta http-equiv="Content-Type" content="text/html; charset=UTF-8">
			<meta name="viewport" content="width=device-width,initial-scale=1">
		<meta name="theme-color" content="#171a21">
		<title>Steam Search</title>
	<link rel="shortcut icon" href="/favicon.ico" type="image/x-icon">

	
	
	<link href="https://store.akamai.steamstatic.com/public/shared/css/motiva_sans.css?v=4Vj7bvhvv_UL&amp;l=english&amp;_cdn=akamai" rel="stylesheet" type="text/css">
<link href="https://store.akamai.steamstatic.com/public/shared/css/shared_global.css?v=5_x1GtS602xY&amp;l=english&amp;_cdn=akamai" rel="stylesheet" type="text/css">
<link href="https://store.akamai.steamstatic.com/public/shared/css/buttons.css?v=u7ItYmj40jWH&amp;l=english&amp;_cdn=akamai" rel="stylesheet" type="text/css">
<link href="https://store.akamai.steamstatic.com/public/css/v6/store.css?v=yMKjnh9kCiOn&amp;l=english&amp;_cdn=akamai" rel="stylesheet" type="text/css"

In [35]:
soup = BeautifulSoup(html_content, 'html.parser')
# print(soup)
search_results = soup.find_all('a', class_='search_result_row')
print(search_results)

[<a class="search_result_row ds_collapse_flag" data-ds-appid="2483190" data-ds-crtrids="[44451136,3090835]" data-ds-itemkey="App_2483190" data-ds-steam-deck-compat-handled="true" data-ds-tagids="[699,1695,1644,3859,1100687,599,3834]" data-gpnav="item" data-search-page="1" href="https://store.steampowered.com/app/2483190/Forza_Horizon_6/?snr=1_7_7_230_150_1" onmouseout="HideGameHover( this, event, 'global_hover' )" onmouseover="GameHover( this, event, 'global_hover', {&quot;type&quot;:&quot;app&quot;,&quot;id&quot;:2483190,&quot;public&quot;:1,&quot;v6&quot;:1} );">
<div class="search_capsule"><img src="https://shared.akamai.steamstatic.com/store_item_assets/steam/apps/2483190/d0d73e4ff499d6cd6fb06f7af898414179997125/capsule_231x87.jpg?t=1779221600"/></div>
<div class="responsive_search_name_combined">
<div class="search_name ellipsis">
<span class="title">Forza Horizon 6</span>
</div>
<div class="search_platforms">
<span class="platform_img win"></span> </div>
<div class="search_releas

In [36]:
for item in search_results:
    # try:
    # Wyciągamy ID aplikacji (Steam trzyma je w atrybucie 'data-ds-appid')
    app_id = item.get('data-ds-appid')
    
    # Wyciągamy nazwę gry
    name = item.find('span', class_='title').text.strip()
    
    # Szukamy sekcji z opiniami (ma klasą 'search_review_summary')
    review_div = item.find('span', class_='search_review_summary')
    
    if review_div:
        # Wyciągamy ukryty opis tekstowy, np. "Mixed - 45% of the 1,200 user reviews..."
        review_html = review_div.get('data-tooltip-html', '')
        
        # Używamy Wyrażeń Regularnych (Regex), żeby wyciągnąć procenty i liczbę opinii
        # Szukamy wzorca: "XX% of the XX,XX user reviews"
        percent_grp = re.findall(r'(\d\d)%', review_html)
        reviews_grp = re.findall(r'of the ([\d,]+) user reviews', review_html)
        
        review_percent = int(percent_grp[0]) if percent_grp else None
        
        # Usuwamy przecinki z liczby opinii, np. 1,200 -> 1200
        review_count = int(reviews_grp[0].replace(',', '')) if reviews_grp else 0
        print(f"{app_id} - {name} - {review_percent} - {review_count}")
    # except Exception as e:
    #     # Jeśli jedna gra rzuci błędem podczas parsowania, idziemy do kolejnej
    #     continue

2483190 - Forza Horizon 6 - 86 - 19699
730 - Counter-Strike 2 - 86 - 2542071
1962700 - Subnautica 2 - 93 - 52658
2215200 - LEGO® Batman™: Legacy of the Dark Knight - 96 - 2642
3041230 - Windrose - 89 - 24154
3321460 - Crimson Desert - 86 - 56162
3892270 - Gamble With Your Friends - 88 - 5070
3240220 - Grand Theft Auto V Enhanced - 79 - 61236
3405690 - EA SPORTS FC™ 26 - 47 - 18187
264710 - Subnautica - 96 - 182006
3124540 - Far Far West - 97 - 16551
3105440 - Heroes of Might and Magic: Olden Era - 88 - 4748
227300 - Euro Truck Simulator 2 - 97 - 141308
2183900 - Warhammer 40,000: Space Marine 2 - 86 - 91089
230410 - Warframe - 91 - 293413
1172470 - Apex Legends™ - 76 - 442638
2605790 - Deep Rock Galactic: Rogue Core - 70 - 4151
578080 - PUBG: BATTLEGROUNDS - 66 - 445683
1174180 - Red Dead Redemption 2 - 91 - 298457
359550 - Tom Clancy's Rainbow Six Siege - 82 - 781498
2767030 - Marvel Rivals - 77 - 281547
236390 - War Thunder - 74 - 311058
4197610 - Librarian: Tidy Up the Arcane Librar